# Quasi-BRDF drift workflow — all PICS (pooled multi-site)

Multi-site version of `BRDF_drift_workflow_Libya4_10_7_1.ipynb`. Every PICS
contributes **one representative patch = one site**; all sites share a **single
drift rate** while each keeps its own brightness/BRDF terms. For `S` sites the
state vector is `2S + 1`:

$$\hat\rho_{s} = (\alpha_s + \beta_s\,\cos\theta)\,(1 + \gamma\, t), \qquad
\mathbf{x} = [\alpha_1..\alpha_S,\ \beta_1..\beta_S,\ \gamma]$$

`α_s`, `β_s` are per-site nuisances (each pinned by its own scenes); `γ` is the
one shared instrument drift, constrained by **all** sites jointly. `t` is years
since the earliest scene across all sites (a common clock), so `γ` is one rate.

**Why pool:** the extra per-site parameters are cheap locals, while the added
scenes — and especially their *geometric diversity* — sharpen and de-confound the
single shared `γ`. This only holds if the drift is genuinely common across sites,
so Step 4 checks each site's standalone `γ` for mutual consistency before we trust
the pooled value.

**Exclusions / inclusion rule:** Mauritania 2 is excluded — its patch (`10_6_7`)
has only 2 QC-surviving scenes, which exactly saturate its own `α/β` and give
**zero** leverage on `γ`. A general `MIN_SCENES` floor (default 4, post-QC)
enforces this for every site.

**Scope:** one channel (default 858 nm, editable), rad and rfl fit on each site's
identical QC-surviving ∩ `has_rfl` scene set.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm
from scipy.optimize import least_squares


def find_repo_root(start=None):
    """Walk up from the notebook's cwd until the emit_pics/ package is found."""
    p = (start or Path.cwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / 'emit_pics').is_dir():
            return cand
    raise RuntimeError('Could not locate repo root (no emit_pics/ upward from cwd).')


REPO = find_repo_root()
sys.path.insert(0, str(REPO))

from emit_pics.config import load_sites
from emit_pics.qc_gate import apply_qc_filter, QCReviewRequired
from emit_pics.delta_calculations import load_scene_spectrum

# ---------------- USER-CONFIGURABLE INPUTS ----------------
# One representative patch per PICS (one patch = one "site" in the model).
# Fill a patch id for each site to include; leave None to skip it (the roster
# cell lists that site's available patches to help you choose).
SITE_PATCHES = {
    'Libya_4':      '10_7_1',
    'Algeria_3':    None,
    'Algeria_5':    None,
    'Libya_1':      None,
    'Mauritania_1': None,   # the ~10-scene patch
}

# Mauritania 2 excluded by design: its selected patch (10_6_7) has only 2
# QC-surviving scenes, which exactly saturate its own alpha/beta and so
# contribute nothing to the shared drift gamma (see caveats at the end).
EXCLUDE_SITES = {'Mauritania_2'}

TARGET_WL         = 858.0   # nm; nearest channel is used (editable)
MIN_SCENES        = 4       # per-site post-QC floor; sites below are dropped
REQUIRE_QC_REVIEW = True    # require a reviewed QC worksheet before fitting
N_BOOTSTRAP       = 1000    # resamples for the pooled gamma uncertainty
PER_SITE_BOOT     = 500     # resamples for each site's standalone gamma CI

DATA      = REPO / 'data'
GEOM_DIR  = DATA / 'geometry'
QC_DIR    = DATA / 'qc'
RFL_DIR   = DATA / 'reflectance'
RAD_DIR   = DATA / 'radiance'

PRODUCTS  = {'rad': '#1f77b4', 'rfl': '#2ca02c'}   # product -> plot color
YR_SEC    = 365.25 * 86400
print('repo root:', REPO)

## 1. Resolve the site roster

Take every PICS in `config/sites.yaml`, drop `EXCLUDE_SITES`, and map each to its
representative patch from `SITE_PATCHES`. Sites left unmapped are reported (with
their available patch geometries) and skipped, so the notebook runs on whatever
you have filled in.

In [ ]:
# site_name 'Libya 4' -> data id 'Libya_4' (matches the geometry/qc filenames).
all_sites = [info['site_name'].replace(' ', '_') for info in load_sites().values()]

ROSTER, unmapped = {}, []
for pics_id in all_sites:
    if pics_id in EXCLUDE_SITES:
        continue
    patch = SITE_PATCHES.get(pics_id)
    (ROSTER.__setitem__(pics_id, patch) if patch else unmapped.append(pics_id))

print('Included sites (%d):' % len(ROSTER))
for s, p in ROSTER.items():
    print(f'  {s:<14} patch {p}')
if EXCLUDE_SITES:
    print('Excluded by config:', ', '.join(sorted(EXCLUDE_SITES)))

for s in unmapped:
    cands = sorted(q.name[len(f'geometry_{s}_'):-4]
                   for q in GEOM_DIR.glob(f'geometry_{s}_*.csv'))
    print(f'\n[skip] {s}: no patch set in SITE_PATCHES. '
          f'Available patches: {cands if cands else "(none found)"}')

if not ROSTER:
    raise SystemExit('No sites mapped. Fill SITE_PATCHES with at least one patch id.')

## 2. Assemble the pooled per-scene table

Reuse the single-patch recipe per site — geometry table + QC gate + restrict to
QC-surviving scenes that also carry reflectance (`has_rfl`, so rad and rfl share
one scene set per site), then per-scene median ρ at the band nearest `TARGET_WL`.
Scenes are stacked with a `site` label. `t` is set **globally** (years since the
earliest scene across all sites) so `γ` is one shared drift; `g = cos(SZA_mean)`.

In [ ]:
def load_site_obs(pics_id, patch):
    """Per-scene obs for one site: QC-surviving, has_rfl, median rho at the band.

    Returns (DataFrame | None, wl_used | None, note). The DataFrame has one row
    per usable scene: site, patch, timestamp, datetime, sza_mean, rho_rad, rho_rfl.
    """
    geom_path = GEOM_DIR / f'geometry_{pics_id}_{patch}.csv'
    qc_path   = QC_DIR / f'{pics_id}_{patch}_qc.csv'
    if not geom_path.exists():
        return None, None, f'missing geometry {geom_path.name}'

    geom = pd.read_csv(geom_path)
    geom['datetime'] = pd.to_datetime(geom['datetime'])
    geom['has_rfl'] = (geom['has_rfl'].astype(str).str.strip().str.lower()
                       .isin(['true', '1']))
    geom = geom.sort_values('datetime').reset_index(drop=True)
    n_all = len(geom)

    try:
        geom, _ = apply_qc_filter(geom, qc_path, require_review=REQUIRE_QC_REVIEW)
    except QCReviewRequired as exc:
        return None, None, f'QC not reviewed ({exc})'
    geom = geom[geom['has_rfl']].reset_index(drop=True)
    if geom.empty:
        return None, None, 'no QC-surviving scenes with reflectance'

    ref = load_scene_spectrum(geom.loc[0, 'scene_id'], pics_id, patch, RFL_DIR, RAD_DIR)
    if ref is None:
        return None, None, 'first surviving scene has no radiance CSV'
    bidx = int((ref['central_wl'] - TARGET_WL).abs().idxmin())
    wl_used = float(ref.loc[bidx, 'central_wl'])

    recs = []
    for _, s in geom.iterrows():
        spec = load_scene_spectrum(s['scene_id'], pics_id, patch, RFL_DIR, RAD_DIR)
        if spec is None or len(spec) != len(ref):
            continue
        recs.append({'site': pics_id, 'patch': patch,
                     'timestamp': s['timestamp'], 'datetime': s['datetime'],
                     'sza_mean': s['sza_mean'],
                     'rho_rad': spec.loc[bidx, 'median_rad'],
                     'rho_rfl': spec.loc[bidx, 'median_rfl']})
    return pd.DataFrame(recs), wl_used, f'{n_all} scenes -> {len(recs)} usable'


frames, wl_by_site = [], {}
for pics_id, patch in ROSTER.items():
    df, wl_used, note = load_site_obs(pics_id, patch)
    if df is None or df.empty:
        print(f'[drop] {pics_id}: {note}')
        continue
    wl_by_site[pics_id] = wl_used
    frames.append(df)
    print(f'  {pics_id:<14} {note}  (band ~{wl_used:.1f} nm)')

if not frames:
    raise SystemExit('No usable site data assembled.')

obs = pd.concat(frames, ignore_index=True).sort_values('datetime').reset_index(drop=True)
T0 = obs['datetime'].min()                                   # common time origin
obs['t'] = (obs['datetime'] - T0).dt.total_seconds() / YR_SEC
obs['g'] = np.cos(np.radians(obs['sza_mean']))

wl_set = sorted({round(w, 1) for w in wl_by_site.values()})
print(f'\nPooled: {len(obs)} scenes across {obs["site"].nunique()} sites | '
      f't0 = {T0:%Y-%m-%d} | bands {wl_set} nm')

## 3. Per-site scene counts, `MIN_SCENES` gate, and leverage

Each site needs more scenes than its 2 local parameters to inform `γ` at all;
`MIN_SCENES` (default 4) drops the rest and logs them. The table also reports
each site's SZA / time spread and `corr(cos SZA, t)` — a site can clear the count
but still be geometry-limited (`|corr|` near 1 → its own `β`/`γ` are confounded).

In [ ]:
rows = []
for site, sub in obs.groupby('site'):
    rows.append({'site': site, 'n': len(sub),
                 'first': sub['datetime'].min().date(),
                 'last': sub['datetime'].max().date(),
                 'sza_lo': round(sub['sza_mean'].min(), 1),
                 'sza_hi': round(sub['sza_mean'].max(), 1),
                 'cosSZA_span': round(sub['g'].max() - sub['g'].min(), 3),
                 't_span_yr': round(sub['t'].max() - sub['t'].min(), 2),
                 'corr_g_t': (round(np.corrcoef(sub['g'], sub['t'])[0, 1], 2)
                              if len(sub) > 2 else np.nan)})
summary = pd.DataFrame(rows).sort_values('site').reset_index(drop=True)

too_small = summary.loc[summary['n'] < MIN_SCENES, 'site'].tolist()
for s in too_small:
    n = int(summary.loc[summary.site == s, 'n'].iloc[0])
    print(f'[drop] {s}: {n} scenes < MIN_SCENES={MIN_SCENES} '
          '(too few to constrain drift beyond its own alpha/beta).')
if too_small:
    obs = obs[~obs['site'].isin(too_small)].reset_index(drop=True)
    summary = summary[~summary['site'].isin(too_small)].reset_index(drop=True)

SITES = sorted(obs['site'].unique())
print(f'\nFinal fit set: {len(SITES)} sites, {len(obs)} scenes')
summary

## 4a. Pooled leverage / confounding scatter

ρ vs `t` and ρ vs `cos(SZA)`, colored by site. Sites separate vertically because
each has its own brightness (that is exactly what the per-site `α_s` absorbs). The
pooled `corr(cos SZA, t)` printed below is the multi-site analog of the single-
patch confounding check — pooling several cadences should push it toward 0.

In [ ]:
site_colors = {s: cm.tab10(i % 10) for i, s in enumerate(SITES)}
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for i, prod in enumerate(PRODUCTS):
    y = obs[f'rho_{prod}']
    for s in SITES:
        m = obs['site'] == s
        axes[i, 0].scatter(obs.loc[m, 't'], y[m], s=26, color=site_colors[s],
                           edgecolors='k', linewidths=0.3, label=s)
        axes[i, 1].scatter(obs.loc[m, 'g'], y[m], s=26, color=site_colors[s],
                           edgecolors='k', linewidths=0.3)
    axes[i, 0].set_xlabel(f't (yr since {T0.date()})')
    axes[i, 0].set_ylabel(f'{prod} rho (median)')
    axes[i, 1].set_xlabel('cos(SZA)'); axes[i, 1].set_ylabel(f'{prod} rho (median)')
axes[0, 0].set_title('rho vs time'); axes[0, 1].set_title('rho vs cos(SZA)')
axes[0, 0].legend(fontsize=7, ncol=2, title='site')
fig.suptitle(f'All-PICS leverage check — band ~{TARGET_WL:.0f} nm', fontsize=12)
fig.tight_layout()
plt.show()

print(f'Pooled corr(cos SZA, t) = {np.corrcoef(obs["g"], obs["t"])[0, 1]:+.2f} '
      '(near 0 -> drift well separated from geometry across the pooled set)')

## 4b. Per-site standalone γ — shared-drift consistency check

Before collapsing to one `γ`, fit each site independently (its own 3-parameter
model) and bootstrap its `γ` CI. **If the sites share a common drift, these
per-site `γ`'s should be mutually consistent (overlapping CIs).** Wildly scattered
per-site `γ`'s would mean the shared-`γ` assumption is wrong and the pooled value
is a blend of differing true drifts — a red flag, not a result.

In [ ]:
def fit_site(g, t, rho):
    """Standalone 3-parameter fit [alpha, beta, gamma] for one site/channel."""
    g = np.asarray(g, float); t = np.asarray(t, float); rho = np.asarray(rho, float)
    ok = np.isfinite(g) & np.isfinite(t) & np.isfinite(rho)
    g, t, rho = g[ok], t[ok], rho[ok]
    a0, b0 = np.linalg.lstsq(np.column_stack([np.ones(g.size), g]), rho, rcond=None)[0]
    res = least_squares(lambda x: (x[0] + x[1] * g) * (1 + x[2] * t) - rho, [a0, b0, 0.0])
    return {'alpha': res.x[0], 'beta': res.x[1], 'gamma': res.x[2], 'n': int(g.size)}


def boot_site_gamma(g, t, rho, n_boot, seed=1):
    g = np.asarray(g, float); t = np.asarray(t, float); rho = np.asarray(rho, float)
    ok = np.isfinite(g) & np.isfinite(t) & np.isfinite(rho)
    g, t, rho = g[ok], t[ok], rho[ok]
    base = fit_site(g, t, rho)
    x0 = [base['alpha'], base['beta'], base['gamma']]
    rng = np.random.default_rng(seed)
    n, draws = g.size, np.empty(n_boot)
    for k in range(n_boot):
        idx = rng.integers(0, n, n)
        r = least_squares(
            lambda x: (x[0] + x[1] * g[idx]) * (1 + x[2] * t[idx]) - rho[idx], x0)
        draws[k] = r.x[2]
    return base['gamma'], np.percentile(draws, [2.5, 97.5])


persite = []
for prod in PRODUCTS:
    for s in SITES:
        sub = obs[obs['site'] == s]
        g0, ci = boot_site_gamma(sub['g'], sub['t'], sub[f'rho_{prod}'], PER_SITE_BOOT)
        persite.append({'product': prod, 'site': s, 'n': len(sub),
                        'gamma_pct_yr': 100 * g0,
                        'ci_lo': 100 * ci[0], 'ci_hi': 100 * ci[1]})
persite = pd.DataFrame(persite)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, prod in zip(axes, PRODUCTS):
    d = persite[persite['product'] == prod].reset_index(drop=True)
    ypos = np.arange(len(d))
    ax.errorbar(d['gamma_pct_yr'], ypos,
                xerr=[d['gamma_pct_yr'] - d['ci_lo'], d['ci_hi'] - d['gamma_pct_yr']],
                fmt='o', color=PRODUCTS[prod], ecolor='0.5', capsize=3)
    ax.axvline(0, color='k', lw=0.8, ls='--')
    ax.set_yticks(ypos); ax.set_yticklabels(d['site'])
    ax.set_xlabel('per-site gamma (%/yr)'); ax.set_title(prod.upper())
fig.suptitle('Per-site standalone drift (shared-gamma check) — '
             'overlapping CIs support one common gamma', fontsize=11)
fig.tight_layout()
plt.show()
persite.round(3)

## 5. Joint pooled fit — per-site α, β with one shared γ

Stack all sites into a single nonlinear least-squares fit of the `2S + 1` vector,
warm-started from each site's OLS of ρ vs `cos(SZA)` (`α₀`, `β₀`) with `γ₀ = 0`.
Only `γ` is shared; every `α_s`, `β_s` is local.

In [ ]:
def _build(frame, prod):
    sites = sorted(frame['site'].unique())
    lut = {s: i for i, s in enumerate(sites)}
    sidx = frame['site'].map(lut).to_numpy()
    g = frame['g'].to_numpy(float); t = frame['t'].to_numpy(float)
    rho = frame[f'rho_{prod}'].to_numpy(float)
    ok = np.isfinite(g) & np.isfinite(t) & np.isfinite(rho)
    return sites, sidx[ok], g[ok], t[ok], rho[ok]


def _resid(x, S, sidx, g, t, rho):
    a = x[:S][sidx]; b = x[S:2 * S][sidx]; ga = x[2 * S]
    return (a + b * g) * (1 + ga * t) - rho


def fit_joint(frame, prod):
    sites, sidx, g, t, rho = _build(frame, prod)
    S = len(sites)
    a0, b0 = np.empty(S), np.empty(S)
    for i in range(S):
        m = sidx == i
        a0[i], b0[i] = np.linalg.lstsq(
            np.column_stack([np.ones(m.sum()), g[m]]), rho[m], rcond=None)[0]
    x0 = np.concatenate([a0, b0, [0.0]])
    res = least_squares(_resid, x0, args=(S, sidx, g, t, rho))
    return {'sites': sites, 'alpha': res.x[:S], 'beta': res.x[S:2 * S],
            'gamma': float(res.x[2 * S]), 'success': bool(res.success),
            'sidx': sidx, 'labels': np.array(sites)[sidx],
            'g': g, 't': t, 'rho': rho, 'n': int(g.size)}


joint = {p: fit_joint(obs, p) for p in PRODUCTS}

for prod, r in joint.items():
    print(f'== {prod.upper()} ==  shared gamma = {100 * r["gamma"]:+.3f} %/yr   '
          f'(n={r["n"]}, {len(r["sites"])} sites, success={r["success"]})')
    for s, a, b in zip(r['sites'], r['alpha'], r['beta']):
        print(f'    {s:<14} alpha={a:>12.4f}  beta={b:>12.4f}')

## 6. Quasi-BRDF correction — divide out each site's factor

Dividing every scene by its own fitted `(α_s + β_s·cosθ)` removes brightness and
geometry, leaving the shared `(1 + γt)`. If the model holds, all sites collapse
onto one drift line regardless of how different their raw ρ levels were.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6), sharex=True)
tg = np.linspace(float(obs['t'].min()), float(obs['t'].max()), 60)
for ax, prod in zip(axes, PRODUCTS):
    r = joint[prod]
    a = r['alpha'][r['sidx']]; b = r['beta'][r['sidx']]
    corrected = r['rho'] / (a + b * r['g'])
    for s in r['sites']:
        m = r['labels'] == s
        ax.scatter(r['t'][m], corrected[m], s=24, color=site_colors.get(s, 'gray'),
                   edgecolors='k', linewidths=0.3, label=s, zorder=3)
    ax.plot(tg, 1 + r['gamma'] * tg, 'k-', lw=1.8, zorder=4,
            label=f"1 + gamma*t ({100 * r['gamma']:+.3f} %/yr)")
    ax.axhline(1.0, color='k', lw=0.6, ls=':')
    ax.set_title(f'{prod.upper()} — corrected'); ax.set_xlabel(f't (yr since {T0.date()})')
axes[0].set_ylabel(r'$\rho\,/\,(\alpha_s + \beta_s\,\cos\theta)$')
axes[0].legend(fontsize=7, ncol=2)
fig.suptitle(f'All-PICS quasi-BRDF correction — band ~{TARGET_WL:.0f} nm: '
             'per-site factor divided out, shared drift retained', fontsize=12)
fig.tight_layout()
plt.show()

## 7. Bootstrap the shared γ (stratified within-site)

Resample scenes **with replacement within each site** (stratified — every site
keeps its scene count), refit the full joint model, keep `γ`. This captures
between-scene scatter while preserving the multi-site design. The percentile CI is
the 2.5/97.5 pctiles of the pooled-`γ` draws.

*Alternative not used here:* resampling whole sites would fold in between-site
variance too, but with only a handful of sites that bootstrap is far too coarse.

In [ ]:
def bootstrap_joint_gamma(frame, prod, n_boot, seed=42):
    base = fit_joint(frame, prod)
    S, sidx = len(base['sites']), base['sidx']
    g, t, rho = base['g'], base['t'], base['rho']
    x0 = np.concatenate([base['alpha'], base['beta'], [base['gamma']]])
    pos = [np.where(sidx == i)[0] for i in range(S)]     # stratified by site
    rng = np.random.default_rng(seed)
    draws = np.empty(n_boot)
    for k in range(n_boot):
        pick = np.concatenate([rng.choice(p, size=p.size, replace=True) for p in pos])
        r = least_squares(_resid, x0, args=(S, sidx[pick], g[pick], t[pick], rho[pick]))
        draws[k] = r.x[2 * S]
    return {'gamma': base['gamma'], 'se': float(draws.std()),
            'ci': np.percentile(draws, [2.5, 97.5]), 'draws': draws}


boots = {p: bootstrap_joint_gamma(obs, p, N_BOOTSTRAP) for p in PRODUCTS}

print(f'{"":<8}{"gamma %/yr":>12}{"boot SE":>10}{"95% CI (%/yr)":>24}'
      f'{"n":>6}{"sites":>7}')
for p, b in boots.items():
    lo, hi = 100 * b['ci']
    print(f'{p:<8}{100 * b["gamma"]:>+12.3f}{100 * b["se"]:>10.3f}'
          f'{f"[{lo:+.3f}, {hi:+.3f}]":>24}{joint[p]["n"]:>6}{len(joint[p]["sites"]):>7}')

fig, ax = plt.subplots(figsize=(7.5, 3.4))
for p, color in PRODUCTS.items():
    ax.hist(100 * boots[p]['draws'], bins=40, color=color, alpha=0.55, label=p)
    ax.axvline(100 * joint[p]['gamma'], color=color, lw=1.6)
ax.axvline(0.0, color='k', lw=0.8, ls='--')
ax.set_xlabel('shared gamma (%/yr)'); ax.set_ylabel('bootstrap count')
ax.set_title(f'All-PICS pooled drift bootstrap — band ~{TARGET_WL:.0f} nm '
             '(stratified within-site)')
ax.legend()
fig.tight_layout()
plt.show()

### Reading this / caveats

* **γ** is the pooled single-channel drift (%/yr) shared across all included
  PICS, from the full scene pool (no geometry-matching). A 95% CI straddling 0
  means no significant drift detected at this band.
* **Trust the pooled γ only if Step 4b was clean** — the per-site γ's mutually
  consistent (overlapping CIs). If they scatter, the shared-γ premise is broken
  and the pooled number is a weighted blend, not a drift estimate.
* **Inclusion rule:** `MIN_SCENES` (post-QC) keeps saturated sites out; Mauritania
  2 is excluded on this basis. Raising the floor trades sites for robustness.
* **rad vs rfl** use the identical scene set per site, so a γ difference is a
  product effect (atmosphere / ISOFIT), not different sampling.
* **Bootstrap** is stratified within-site case resampling → between-scene variance
  only; within-patch spatial spread and instrument uncertainty are not included.
  Percentile CI (not BCa) — fine when the draws are roughly symmetric.
* **Next steps:** sweep all channels into a γ(λ) drift spectrum (mask the same
  water-vapor bands for both products); optionally move to partial pooling
  (hierarchical α/β) so sparse sites like Mauritania 2 could re-enter legitimately.
  Change `TARGET_WL` above to inspect a different channel.

## 8. γ(λ) drift spectrum — pooled shared-γ per channel

Repeat the pooled joint fit **at every EMIT channel** (water-vapor absorption
bands excluded, for both products), and plot the shared-γ best estimate with its
bootstrap 95% CI as whiskers vs wavelength — the multi-site analog of the
matched-pairs drift spectrum.

**Multiple comparisons.** With ~244 channels, testing each for γ≠0 at α=0.05 would
flag ≈12 "significant" channels by chance alone. So significance is decided by
**Benjamini–Hochberg FDR** across channels (per product), identical to the
matched-pairs figure — a channel is red only if it survives FDR < `ALPHA`. Each
channel's p-value is a two-sided **bootstrap** p (twice the smaller tail fraction
of its γ draws about 0).

*Dependence caveat:* neighbouring channels are highly correlated, so the tests are
not independent. BH controls FDR under positive dependence (PRDS), which spectral
smoothness approximately satisfies; if you want a guarantee under arbitrary
dependence, swap in Benjamini–Yekutieli (more conservative). *Runtime:* this fits
2 × (#channels) × (`SPECTRUM_N_BOOT`+1) small models — lower `SPECTRUM_N_BOOT` for
a quick look, raise it for publication.

In [ ]:
from emit_pics.delta_calculations import _perchannel_path
from emit_pics.matched_pairs_drift_analysis import benjamini_hochberg

SPECTRUM_N_BOOT = 300      # per-channel bootstrap resamples (heavier than Step 7)
ALPHA           = 0.05     # significance level (BH-FDR and the raw comparison)


def _scene_perchannel(row, product):
    d = RAD_DIR if product == 'rad' else RFL_DIR
    path = _perchannel_path(d, row['site'], row['patch'], str(row['timestamp']), product)
    return pd.read_csv(path) if path.exists() else None


# Reference grid + water-vapor exclusion. EMIT L2A reflectance is fill (-> NaN) in
# the deep H2O windows; that is a fixed instrument property, so one reflectance-
# bearing scene defines the excluded-band mask for BOTH products.
ref = _scene_perchannel(obs.iloc[0], 'rfl')
WLGRID = ref['central_wl'].to_numpy(float)
NBAND  = len(WLGRID)
EXCL   = ~np.isfinite(ref['median_rfl'].to_numpy(float))     # excluded (water-vapor) bands
FIT_BANDS = np.where(~EXCL)[0]
print(f'{NBAND} channels | {int(EXCL.sum())} water-vapor bands excluded | '
      f'{len(FIT_BANDS)} fit per product')


def _matrix(product):
    """rows = obs scenes, cols = bands; median rho, band-order aligned."""
    M = np.full((len(obs), NBAND), np.nan)
    for i, (_, row) in enumerate(obs.iterrows()):
        s = _scene_perchannel(row, product)
        if s is not None and len(s) == NBAND:
            M[i] = s[f'median_{product}'].to_numpy(float)
    return M


MAT      = {p: _matrix(p) for p in PRODUCTS}
g_all    = obs['g'].to_numpy(float)
t_all    = obs['t'].to_numpy(float)
site_all = obs['site'].to_numpy()


def _warm(S, sidx, g, t, rho):
    a0, b0 = np.zeros(S), np.zeros(S)
    for i in range(S):
        m = sidx == i
        if m.sum() >= 2:
            a0[i], b0[i] = np.linalg.lstsq(
                np.column_stack([np.ones(m.sum()), g[m]]), rho[m], rcond=None)[0]
        elif m.sum() == 1:
            a0[i] = rho[m].mean()
    return np.concatenate([a0, b0, [0.0]])


def _channel_gamma(rho, n_boot, rng):
    """Pooled shared-gamma at one channel + bootstrap CI and two-sided boot p."""
    ok = np.isfinite(rho) & np.isfinite(g_all) & np.isfinite(t_all)
    g, t, r = g_all[ok], t_all[ok], rho[ok]
    present, si = np.unique(site_all[ok], return_inverse=True)   # remap present sites
    S = len(present)
    if S == 0 or r.size <= 2 * S:
        return np.nan, (np.nan, np.nan), np.nan, int(r.size)
    base = least_squares(_resid, _warm(S, si, g, t, r), args=(S, si, g, t, r))
    gam = base.x[2 * S]
    pos = [np.where(si == i)[0] for i in range(S)]
    draws = np.empty(n_boot)
    for k in range(n_boot):
        pick = np.concatenate([rng.choice(p, size=p.size, replace=True) for p in pos])
        res = least_squares(_resid, base.x, args=(S, si[pick], g[pick], t[pick], r[pick]))
        draws[k] = res.x[2 * S]
    ci = np.percentile(draws, [2.5, 97.5])
    npos, nneg = int((draws > 0).sum()), int((draws < 0).sum())
    pval = min(1.0, 2 * min((npos + 1) / (n_boot + 1), (nneg + 1) / (n_boot + 1)))
    return gam, (ci[0], ci[1]), pval, int(r.size)


rows, rng = [], np.random.default_rng(2024)
for prod in PRODUCTS:
    for j, b in enumerate(FIT_BANDS):
        gam, (lo, hi), pval, n = _channel_gamma(MAT[prod][:, b], SPECTRUM_N_BOOT, rng)
        rows.append({'product': prod, 'band_index': int(b),
                     'wavelength': float(WLGRID[b]), 'gamma_pct_yr': 100 * gam,
                     'ci_lo_pct_yr': 100 * lo, 'ci_hi_pct_yr': 100 * hi,
                     'p_value': pval, 'n': n})
        if j % 40 == 0:
            print(f'  {prod}: channel {j + 1}/{len(FIT_BANDS)} ({WLGRID[b]:.0f} nm)')
spectrum = pd.DataFrame(rows)

# Significance two ways: uncorrected raw p, and BH-FDR across channels (per product).
spectrum['q_value'] = np.nan
spectrum['sig_raw'] = spectrum['p_value'] <= ALPHA        # UNCORRECTED
spectrum['sig_bh']  = False                               # BH-FDR corrected
for prod in PRODUCTS:
    m = (spectrum['product'] == prod).to_numpy()
    reject, q = benjamini_hochberg(spectrum.loc[m, 'p_value'].to_numpy(), ALPHA)
    spectrum.loc[m, 'q_value'] = q
    spectrum.loc[m, 'sig_bh'] = reject
    print(f'{prod}: raw p<{ALPHA}: {int(spectrum.loc[m, "sig_raw"].sum())}  |  '
          f'BH FDR<{ALPHA}: {int(reject.sum())}   (of {int(m.sum())} channels)')

# Flag numerically unstable channels. The multiplicative model (alpha+beta*g)(1+gamma*t)
# expresses gamma as a change *relative to the baseline* (alpha+beta*g). Where that
# baseline approaches zero -- deep-SWIR / low-signal radiance bands -- gamma explodes to
# absurd values with an enormous CI. These are fit degeneracies, not drift; they never
# read as significant (their CI straddles 0) but they wreck the plot autoscale, so we
# flag them and keep them off the drawn scale.
GAMMA_CAP = 500.0    # %/yr; |gamma| or CI half-width beyond this = degenerate fit
_ci_hw = (spectrum['ci_hi_pct_yr'] - spectrum['ci_lo_pct_yr']).abs() / 2
spectrum['unstable'] = (~np.isfinite(spectrum['gamma_pct_yr'])
                        | (spectrum['gamma_pct_yr'].abs() > GAMMA_CAP)
                        | (_ci_hw > GAMMA_CAP))
for prod in PRODUCTS:
    nu = int(spectrum.loc[spectrum['product'] == prod, 'unstable'].sum())
    if nu:
        print(f'{prod}: {nu} unstable channel(s) (near-zero baseline, '
              f'|gamma| or CI half-width > {GAMMA_CAP:.0f} %/yr) hidden from the plot scale')
spectrum.head()

In [ ]:
# Manual y-limits (%/yr) for the two panels: set to a (low, high) tuple to override
# the autoscale, or leave None to autoscale to the data.
RAD_YLIM = None    # e.g. (-60, 60)
RFL_YLIM = None    # e.g. (-20, 15)


def plot_drift_spectrum(sig_col, sig_label, title_tag):
    """Two-panel drift spectrum (rad top, rfl bottom); red = *sig_col* True."""
    def _runs(mask):
        runs, i, n = [], 0, len(mask)
        while i < n:
            if mask[i]:
                j = i
                while j + 1 < n and mask[j + 1]:
                    j += 1
                runs.append((i, j)); i = j + 1
            else:
                i += 1
        return runs

    wv_runs = _runs(EXCL)
    nfit = int((spectrum['product'] == 'rad').sum())
    fig, axes = plt.subplots(2, 1, figsize=(15, 9), sharex=True)
    for ax, prod, ylab in zip(
            axes, ['rad', 'rfl'],
            ['Radiance (L1B)\ndrift (%/yr)', 'Reflectance (L2A)\ndrift (%/yr)']):
        for k, (a, b) in enumerate(wv_runs):
            ax.axvspan(WLGRID[a], WLGRID[b], color='0.85', zorder=0,
                       label='water-vapor band (excluded)' if k == 0 else None)
        d = spectrum[(spectrum['product'] == prod)
                     & (~spectrum['unstable'])].sort_values('wavelength')
        yerr = np.vstack([d['gamma_pct_yr'] - d['ci_lo_pct_yr'],
                          d['ci_hi_pct_yr'] - d['gamma_pct_yr']])
        ax.errorbar(d['wavelength'], d['gamma_pct_yr'], yerr=yerr, fmt='none',
                    ecolor='0.6', elinewidth=0.8, capsize=1.5, zorder=1, label='95% CI')
        ns, sg = d[~d[sig_col]], d[d[sig_col]]
        ax.scatter(ns['wavelength'], ns['gamma_pct_yr'], s=16, facecolors='none',
                   edgecolors='0.35', linewidths=0.8, zorder=3,
                   label='drift rate — not significant')
        ax.scatter(sg['wavelength'], sg['gamma_pct_yr'], s=18, color='red', zorder=4,
                   label=f'drift rate — {sig_label}')
        ax.axhline(0.0, color='k', lw=1.0)
        ax.set_ylabel(ylab)
        ylim = RAD_YLIM if prod == 'rad' else RFL_YLIM
        if ylim is not None:
            ax.set_ylim(*ylim)
        nhidden = int(((spectrum['product'] == prod) & spectrum['unstable']).sum())
        if nhidden:
            ax.text(0.99, 0.03, f'{nhidden} unstable channel(s) hidden '
                    '(near-zero baseline)', transform=ax.transAxes, ha='right',
                    va='bottom', fontsize=7, color='0.4')
        ax.legend(fontsize=8, loc='best')
    axes[1].set_xlabel('Wavelength (nm)')
    axes[0].set_title(
        f'Radiometric drift spectrum — all PICS (pooled, {len(SITES)} sites) — {title_tag}   '
        f'(marker = pooled shared-γ, whisker = bootstrap 95% CI; {nfit} channels fit)',
        fontsize=12)
    fig.tight_layout()
    plt.show()


# (1) WITH Benjamini–Hochberg FDR correction (the defensible version at this many channels)
plot_drift_spectrum('sig_bh', f'BH-significant (FDR < {ALPHA})',
                    'WITH Benjamini–Hochberg FDR correction')

# (2) WITHOUT correction — raw per-channel p, for comparison only. Expect ~ALPHA x
#     (#channels) false positives here; that inflation is exactly what BH removes.
plot_drift_spectrum('sig_raw', f'raw-significant (p < {ALPHA}, UNCORRECTED)',
                    'WITHOUT correction (raw per-channel p-values)')